# Does fine-tuning help? It depends on a ratio.**No GPU. No network. About thirty seconds.** This notebook recomputes everynumber in the README from the three CSV files in `data/`, which are the rawper-measurement output of the experiment notebooks. Nothing here is hard-coded:if you change the data, the numbers change.**The quantity.** For every (dataset, episode length, context length) we recordthe log-loss of the fine-tuned model and of the un-adapted model **at the samecontext length**, and report the relative gain    relative gain = (log-loss of base - log-loss of fine-tuned) / log-loss of baseRelative, because absolute log-loss ranges from 0.03 to 0.73 across these 24tasks and a raw difference would be dominated by the easy ones.**The variable of interest.**    ratio = inference context rows / fine-tuning episode context rowsThree independent runs: seed 42, seed 43, and a fixed-query control in which thenumber of supervised query rows is held at 128 for every episode length.

In [1]:

import numpy as np, pandas as pd
from math import comb
from pathlib import Path

DATA = Path('data') if Path('data').is_dir() else Path('../data')
RUNS = {'seed 42': 'curves_seed42.csv',
        'seed 43': 'curves_seed43.csv',
        'fixed-query': 'curves_fixedquery.csv'}
STEPS = 200            # the budget the README reports; 25 and 50 are also in the data

def load(path, label):
    d = pd.read_csv(path)
    for col, default in [('regime', 'standard'), ('ft_steps', 0), ('ep_query', 0)]:
        if col not in d.columns:
            d[col] = default
    d = d[d.regime == 'standard']
    keys = ['target', 'name', 'n_train', 'arm', 'ft_steps', 'ep_context', 'context_size']
    agg = (d.groupby(keys, as_index=False)
             .agg(logloss=('logloss', 'mean'), auc=('auc', 'mean'),
                  ctx=('n_context_rows', 'mean'), repeats=('repeat', 'count')))
    agg['run'] = label
    return agg

RAW = pd.concat([load(DATA / f, k) for k, f in RUNS.items()], ignore_index=True)

base = (RAW[RAW.arm == 'base'][['run', 'target', 'context_size', 'logloss', 'auc']]
        .rename(columns={'logloss': 'll0', 'auc': 'auc0'}))
G = RAW[RAW.arm != 'base'].merge(base, on=['run', 'target', 'context_size'], how='inner')
G['rel'] = 100 * (G.ll0 - G.logloss) / G.ll0
G['d_auc'] = G.auc - G.auc0
G['ratio'] = G.ctx / G.ep_context.clip(lower=1)
T = G[G.ft_steps == STEPS].copy()

print(f'{len(RAW):,} aggregated measurements | {RAW.target.nunique()} datasets '
      f'| runs: {sorted(RAW.run.unique())}')
print(f'episode context lengths : {sorted(T.ep_context.unique())}')
print(f'step budgets available  : {sorted(G.ft_steps.unique())}')
print(f'ratios span             : {T.ratio.min():.3f} to {T.ratio.max():.1f}')
print(f'\nanalysing {len(T):,} fine-tuned measurements at {STEPS} steps.')

4,119 aggregated measurements | 24 datasets | runs: ['fixed-query', 'seed 42', 'seed 43']
episode context lengths : [np.int64(102), np.int64(512), np.int64(1638)]
step budgets available  : [np.int64(25), np.int64(50), np.int64(200)]
ratios span             : 0.020 to 165.5

analysing 1,204 fine-tuned measurements at 200 steps.


## 1. The main resultOne value per dataset first, then the mean across datasets - otherwise thedatasets contributing the most grid cells would dominate.

In [2]:

EDGES = [0, .25, .5, 1, 2, 4, 8, np.inf]
NAMES = ['<= 1/4', '1/4 - 1/2', '1/2 - 1', '1 - 2', '2 - 4', '4 - 8', '> 8']
T['band'] = pd.cut(T.ratio, EDGES, labels=NAMES, include_lowest=True)

out = []
for band, g in T.groupby('band', observed=True):
    row = {'ratio band': band}
    for run in RUNS:
        s = g[g.run == run]
        if len(s) < 4:
            row[run] = '--'
            continue
        per = s.groupby('target').rel.mean()
        row[run] = f'{per.mean():+.2f}%  ({int((per > 0).sum())}/{len(per)})'
    out.append(row)
print('Relative log-loss gain by ratio band, per run\n')
print(pd.DataFrame(out).to_string(index=False))

Relative log-loss gain by ratio band, per run

ratio band         seed 42         seed 43     fixed-query
    <= 1/4 +2.45%  (15/18) +1.82%  (14/18) +2.18%  (14/18)
 1/4 - 1/2 +4.39%  (23/24) +4.63%  (22/24) +5.27%  (21/22)
   1/2 - 1 +4.28%  (22/24) +4.09%  (22/24) +5.01%  (21/22)
     1 - 2 +2.35%  (21/24) +1.55%  (20/24) +3.04%  (19/22)
     2 - 4 +1.21%  (15/24) +0.44%  (15/24) +1.85%  (17/22)
     4 - 8  -3.42%  (5/20)  -0.71%  (8/20)  -1.67%  (8/20)
       > 8  -6.85%  (2/18)  -3.61%  (2/18)  -7.11%  (1/18)


### Pooled across the three runs

In [3]:

def pooled(sub, label):
    per = sub.groupby(['run', 'target']).rel.mean()
    auc = sub.groupby(['run', 'target']).d_auc.mean()
    print(f'{label:<34} {per.mean():+6.2f}%  {int((per > 0).sum())}/{len(per)} improve'
          f'   AUC {auc.mean():+.5f}')
    return per, auc

print('README section 3\n')
dia, dia_auc = pooled(T[(T.ratio >= .5) & (T.ratio <= 2)], 'ratio 0.5 - 2  (matched)')
ext, ext_auc = pooled(T[T.ratio > 4], 'ratio > 4      (extrapolating)')

cut = T[(T.ratio >= .5) & (T.ratio <= 2) & (T.n_train > 1000)]
per = cut.groupby(['run', 'target']).rel.mean()
print(f'\nRestricted to the regime den Breejen et al. work in '
      f'(matched ratio, n_train > 1000):\n  {int((per > 0).sum())}/{len(per)} improve, '
      f'{per.mean():+.2f}%. They report 73% on an earlier model.')

README section 3

ratio 0.5 - 2  (matched)            +3.60%  65/70 improve   AUC +0.01631
ratio > 4      (extrapolating)      -4.06%  11/60 improve   AUC -0.00558

Restricted to the regime den Breejen et al. work in (matched ratio, n_train > 1000):
  44/45 improve, +4.08%. They report 73% on an earlier model.


## 2. The master tableOne panel per deployment context, restricted to the dataset-runs present at everyepisode length, so that within a row nothing changes but the episode. This is thetable the README opens with.

In [4]:

EPS = sorted(T.ep_context.unique())
CTX = ['32', '64', '128', '256', '512', '1024', '2048', 'full']

common = None
for ep in EPS:
    s = set(map(tuple, T[T.ep_context == ep][['run', 'target']].drop_duplicates().values))
    common = s if common is None else (common & s)
C = T[[(r, t) in common for r, t in zip(T.run, T.target)]]
print(f'{len(common)} dataset-runs carry every episode length; the table uses only those.\n')

def cell(cs, ep, col):
    g = C[(C.context_size == cs) & (C.ep_context == ep)]
    if len(g) < 6:
        return None
    per = g.groupby(['run', 'target'])[col].mean()
    return float(g.ctx.mean()) / ep, float(per.mean()), int((per > 0).sum()), len(per)

rows = []
for cs in CTX:
    row = {'deploy with': cs}
    best, bestv = None, -9
    for ep in EPS:
        c = cell(cs, ep, 'd_auc')
        if c is None:
            row[f'ep {int(ep)}'] = '--'; continue
        r, v, k, n = c
        row[f'ep {int(ep)}'] = f'{v:+.4f} (r={r:.2f}, {k}/{n})'
        if v > bestv: best, bestv = int(ep), v
    row['winner'] = f'ep {best}'
    rows.append(row)
print('AUC gained by fine-tuning, vs the un-adapted model at the same context\n')
print(pd.DataFrame(rows).to_string(index=False))

print('\n\nSame cells, relative log-loss gain\n')
rows = []
for cs in CTX:
    row = {'deploy with': cs}
    for ep in EPS:
        c = cell(cs, ep, 'rel')
        row[f'ep {int(ep)}'] = '--' if c is None else f'{c[1]:+.2f}%'
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print('\n\nThe ratio of the winning episode, per row:')
wins = []
for cs in CTX:
    cs_cells = [(ep, cell(cs, ep, 'd_auc')) for ep in EPS]
    cs_cells = [(ep, c) for ep, c in cs_cells if c is not None]
    if not cs_cells: continue
    ep, c = max(cs_cells, key=lambda z: z[1][1])
    wins.append(float(round(c[0], 2)))
    print(f'  context {cs:>5} -> episode {int(ep):>5}, ratio {c[0]:.2f}')
print(f'\nWinning ratios: {wins}')
print('The winning column moves right as the context grows, and no single episode')
print('length wins everywhere. But the winner is NOT simply the episode closest to a')
print('fixed ratio: at 128 and at 512 rows of context it is not. See section 4.')

37 dataset-runs carry every episode length; the table uses only those.

AUC gained by fine-tuning, vs the un-adapted model at the same context

deploy with                   ep 102                  ep 512                 ep 1638  winner
         32  +0.0180 (r=0.31, 35/37) +0.0079 (r=0.06, 29/37) +0.0027 (r=0.02, 26/37)  ep 102
         64  +0.0264 (r=0.63, 35/37) +0.0142 (r=0.12, 32/37) +0.0068 (r=0.04, 29/37)  ep 102
        128  +0.0212 (r=1.25, 36/37) +0.0192 (r=0.25, 35/37) +0.0104 (r=0.08, 31/37)  ep 102
        256  +0.0122 (r=2.51, 31/37) +0.0157 (r=0.50, 37/37) +0.0103 (r=0.16, 37/37)  ep 512
        512  +0.0054 (r=5.02, 26/37) +0.0114 (r=1.00, 32/37) +0.0101 (r=0.31, 35/37)  ep 512
       1024 -0.0019 (r=10.04, 15/37) +0.0043 (r=2.00, 30/37) +0.0055 (r=0.63, 30/37) ep 1638
       2048  -0.0094 (r=20.08, 8/37) -0.0042 (r=4.00, 10/37) +0.0002 (r=1.25, 21/37) ep 1638
       full  -0.0088 (r=63.52, 8/37) -0.0048 (r=12.65, 9/37) -0.0024 (r=3.96, 19/37) ep 1638


Same cells, relat

### And along the diagonal the published work walksBoth lengths growing together. Absolute performance rises — which is what theliterature reports — while the gain attributable to fine-tuning shrinks toward zero.

In [5]:

print(f"{'episode':>8} {'context':>9} {'ratio':>7} | {'base AUC':>9} {'tuned AUC':>10} {'gain':>9}")
for ep, cs in [(102, '128'), (512, '512'), (1638, '2048')]:
    g = C[(C.ep_context == ep) & (C.context_size == cs)]
    if len(g) < 6:
        continue
    k = ['run', 'target']
    print(f'{ep:>8} {cs:>9} {float(g.ctx.mean())/ep:>7.2f} | '
          f'{g.groupby(k).auc0.mean().mean():>9.4f} {g.groupby(k).auc.mean().mean():>10.4f} '
          f'{g.groupby(k).d_auc.mean().mean():>+9.5f}')

 episode   context   ratio |  base AUC  tuned AUC      gain
     102       128    1.25 |    0.7368     0.7580  +0.02122
     512       512    1.00 |    0.7755     0.7870  +0.01144
    1638      2048    1.25 |    0.8120     0.8122  +0.00022


## 3. The asymmetryHold the episode length fixed and vary only the context.Watch the `n` column. Only the **1638-row episode** has a constant sample size(37 dataset-runs at every context), because only the larger datasets can carryepisodes that long. That curve is therefore the one immune to compositioneffects, and it is the one to trust when the three disagree. The shorterepisodes lose datasets at the large-context end, where small training sets runout of rows.Read the left of each curve as *waste* and the right as *damage*: the benefitdecays toward zero on the left without turning negative, and goes clearlynegative on the right.

In [6]:

ORDER = ['32', '64', '128', '256', '512', '1024', '2048', 'full']

def curve(ep):
    rows = []
    for cs in ORDER:
        g = T[(T.ep_context == ep) & (T.context_size == cs)]
        if len(g) < 6:
            continue
        per = g.groupby(['run', 'target']).rel.mean()
        auc = g.groupby(['run', 'target']).d_auc.mean()
        rows.append(dict(context=cs, ratio=round(float(g.ratio.mean()), 3),
                         rel=round(float(per.mean()), 2),
                         improved=f'{int((per > 0).sum())}/{len(per)}',
                         d_auc=round(float(auc.mean()), 5), n=len(per)))
    return pd.DataFrame(rows)

for ep in sorted(T.ep_context.unique()):
    c = curve(ep)
    if len(c) < 4:
        continue
    print(f'\nEpisode fixed at {int(ep)} context rows   (n constant = {sorted(set(c.n))})')
    print(c.to_string(index=False))
    print(f'  peak relative log-loss at ratio {c.loc[c.rel.idxmax(), "ratio"]}'
          f' | peak AUC at ratio {c.loc[c.d_auc.idxmax(), "ratio"]}')


Episode fixed at 102 context rows   (n constant = [42, 45, 60, 70])
context  ratio   rel improved    d_auc  n
     32  0.314  4.77    61/70  0.02514 70
     64  0.627  5.44    64/70  0.03133 70
    128  1.255  3.73    61/70  0.02368 70
    256  2.510  1.87    49/70  0.01080 70
    512  5.020 -2.06    27/60  0.00123 60
   1024 10.039 -4.64    16/45 -0.00379 45
   2048 20.078 -4.58     5/42 -0.00888 42
   full 38.116 -5.79     8/70 -0.00961 70
  peak relative log-loss at ratio 0.627 | peak AUC at ratio 0.627



Episode fixed at 512 context rows   (n constant = [42, 45, 54])
context  ratio   rel improved    d_auc  n
     32  0.062  1.39    35/54  0.01115 54
     64  0.125  2.10    42/54  0.01931 54
    128  0.250  3.77    47/54  0.02415 54
    256  0.500  5.45    52/54  0.02066 54
    512  1.000  4.39    47/54  0.01005 54
   1024  2.000  2.84    38/45  0.00480 45
   2048  4.000 -0.03    23/42 -0.00331 42
   full  9.589 -1.43    16/54 -0.00472 54
  peak relative log-loss at ratio 0.5 | peak AUC at ratio 0.25



Episode fixed at 1638 context rows   (n constant = [37])
context  ratio   rel improved    d_auc  n
     32  0.020  0.60    23/37  0.00272 37
     64  0.039  0.30    23/37  0.00682 37
    128  0.078  0.91    28/37  0.01037 37
    256  0.156  1.49    34/37  0.01035 37
    512  0.313  2.07    37/37  0.01012 37
   1024  0.625  1.34    31/37  0.00548 37
   2048  1.250  0.46    29/37  0.00022 37
   full  3.956 -0.88    18/37 -0.00241 37
  peak relative log-loss at ratio 0.313 | peak AUC at ratio 0.078


### The ratio governs; absolute size is second orderIf the ratio were merely a proxy for episode length, curves at different episodelengths would not agree at matched ratio. They do, within a point.

In [7]:

def sign_test(d):
    n = len(d); k = int((d > 0).sum())
    p = sum(comb(n, i) for i in range(0, min(k, n - k) + 1)) / 2 ** n * 2
    return max(k, n - k), n, min(1.0, p)

PAIRS = [(0.25, 102, '32', 512, '128'), (0.50, 102, '64', 512, '256'),
         (1.00, 102, '128', 512, '512'), (2.00, 102, '256', 512, '1024')]
rows = []
for r, epA, csA, epB, csB in PAIRS:
    a = T[(T.ep_context == epA) & (T.context_size == csA)].groupby(['run', 'target']).rel.mean()
    b = T[(T.ep_context == epB) & (T.context_size == csB)].groupby(['run', 'target']).rel.mean()
    common = sorted(set(a.index) & set(b.index))
    if len(common) < 8:
        continue
    diff = b.loc[common] - a.loc[common]
    k, n, p = sign_test(diff)
    rows.append(dict(ratio=r, short_episode=f'{a.loc[common].mean():+.2f}%',
                     long_episode=f'{b.loc[common].mean():+.2f}%',
                     difference=round(float(diff.mean()), 2),
                     sign_test=f'{k}/{n}', p=round(p, 3)))
print('Same ratio, episode length 16x apart (102 vs 512 context rows)\n')
print(pd.DataFrame(rows).to_string(index=False))
print('\nThe ratio moves the gain from +6% to -7%; episode length moves it by ~1 point.')

Same ratio, episode length 16x apart (102 vs 512 context rows)

 ratio short_episode long_episode  difference sign_test     p
  0.25        +4.97%       +3.77%       -1.20     33/54 0.134
  0.50        +6.05%       +5.45%       -0.60     33/54 0.134
  1.00        +4.70%       +4.39%       -0.32     33/54 0.134
  2.00        +2.16%       +2.84%        0.68     26/45 0.371

The ratio moves the gain from +6% to -7%; episode length moves it by ~1 point.


## 4. Controls**Supervision budget.** A longer episode carries more supervised query rows, soat equal steps it sees more targets. The fixed-query run holds the query countat 128 for every episode length. If the effect were supervision, it woulddisappear there.

In [8]:

fq = T[T.run == 'fixed-query']
wide = (fq[fq.ctx >= 1024].pivot_table(index='target', columns='ep_context', values='rel'))
cols = sorted(wide.columns)
if len(cols) >= 2:
    lo, hi = cols[0], cols[-1]
    d = (wide[hi] - wide[lo]).dropna()
    k, n, p = sign_test(d)
    print(f'Fixed-query run, contexts >= 1024 rows:')
    print(f'  episode {int(hi)} minus episode {int(lo)} : {d.mean():+.2f} points, '
          f'{k}/{n} datasets, p = {p:.4f}')
    print('  The effect survives equal supervision: it is the length, not the targets.')

Fixed-query run, contexts >= 1024 rows:
  episode 1638 minus episode 102 : +2.73 points, 13/13 datasets, p = 0.0002
  The effect survives equal supervision: it is the length, not the targets.


**Replication.** The two seeds should agree band by band.

In [9]:

piv = (T.groupby(['band', 'run'], observed=True)
         .apply(lambda g: g.groupby('target').rel.mean().mean(), include_groups=False)
         .unstack())
print(piv.round(2).to_string())
a, b = piv['seed 42'].dropna(), piv['seed 43'].dropna()
common = a.index.intersection(b.index)
print(f'\nseed 42 vs seed 43 across bands: max absolute gap '
      f'{float((a[common] - b[common]).abs().max()):.2f} points, '
      f'same sign everywhere: {bool(((a[common] > 0) == (b[common] > 0)).all())}')

run        fixed-query  seed 42  seed 43
band                                    
<= 1/4            2.18     2.45     1.82
1/4 - 1/2         5.27     4.39     4.63
1/2 - 1           5.01     4.28     4.09
1 - 2             3.04     2.35     1.55
2 - 4             1.85     1.21     0.44
4 - 8            -1.67    -3.42    -0.71
> 8              -7.11    -6.85    -3.61

seed 42 vs seed 43 across bands: max absolute gap 3.24 points, same sign everywhere: True


**Out of sample.** A rule is only worth something if it survives datasets itwas not fitted on. We calibrate "fine-tune iff ratio <= t" on half the datasetsand evaluate on the other half, over 200 random halves.Note what comes out: the threshold is **a plateau, not a point**. Quoting asingle number would be false precision.

In [10]:

grid = np.exp(np.linspace(np.log(1.2), np.log(8.0), 60))
targets = np.array(sorted(T.target.unique()))

def accuracy(t, subset):
    s = T[T.target.isin(subset)]
    return float(((s.ratio <= t) == (s.rel > 0)).mean())

chosen, held_out, always = [], [], []
for seed in range(200):
    rng = np.random.default_rng(seed)
    c = targets.copy(); rng.shuffle(c)
    cal, test = c[:len(c) // 2], c[len(c) // 2:]
    best = max(grid, key=lambda t: accuracy(t, cal))
    chosen.append(best); held_out.append(accuracy(best, test))
    always.append(float((T[T.target.isin(test)].rel > 0).mean()))
chosen, held_out, always = map(np.array, (chosen, held_out, always))

print(f'threshold chosen : median {np.median(chosen):.2f}  '
      f'[q1 {np.percentile(chosen, 25):.2f}, q3 {np.percentile(chosen, 75):.2f}]')
print(f'held-out accuracy: {100*held_out.mean():.1f}% +/- {100*held_out.std():.1f}')
print(f'"always fine-tune": {100*always.mean():.1f}%')
print(f'the rule wins in {int((held_out > always).sum())} of 200 splits')

print('\nAccuracy is flat across the plateau, which is why we quote a band:')
for t in (1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0):
    wrong = T[(T.ratio <= t) & (T.rel <= 0)]
    print(f'  threshold {t:>4} : accuracy {100*accuracy(t, targets):.1f} %   '
          f'wrongly recommends fine-tuning {len(wrong):>3} times')

threshold chosen : median 4.07  [q1 2.51, q3 4.34]
held-out accuracy: 77.1% +/- 2.7
"always fine-tune": 67.6%
the rule wins in 200 of 200 splits

Accuracy is flat across the plateau, which is why we quote a band:
  threshold  1.5 : accuracy 73.0 %   wrongly recommends fine-tuning 125 times
  threshold  2.0 : accuracy 75.0 %   wrongly recommends fine-tuning 142 times
  threshold  2.5 : accuracy 75.2 %   wrongly recommends fine-tuning 147 times
  threshold  3.0 : accuracy 77.4 %   wrongly recommends fine-tuning 175 times
  threshold  4.0 : accuracy 77.5 %   wrongly recommends fine-tuning 200 times
  threshold  5.0 : accuracy 77.4 %   wrongly recommends fine-tuning 205 times
  threshold  6.0 : accuracy 76.7 %   wrongly recommends fine-tuning 242 times
  threshold  8.0 : accuracy 75.7 %   wrongly recommends fine-tuning 259 times


## 5. The honest negative resultEverything above compares against the un-adapted model **at the same context**.A practitioner is not bound by that: they can use all of their data as context.Against *that* baseline, does the best fine-tuned configuration win?

In [11]:

ref = (RAW[(RAW.arm == 'base') & (RAW.context_size == 'full')]
       [['run', 'target', 'logloss', 'auc']]
       .rename(columns={'logloss': 'll_full', 'auc': 'auc_full'}))
ft = RAW[(RAW.arm != 'base') & (RAW.ft_steps == STEPS)].merge(
    ref, on=['run', 'target'], how='inner')
best = ft.loc[ft.groupby(['run', 'target']).logloss.idxmin()]
gain = (best.ll_full - best.logloss) / best.ll_full

print(f'Best fine-tuned setting vs the base model at FULL context:')
print(f'  {100*gain.mean():+.2f}% relative log-loss, winning {int((gain>0).sum())}/{len(gain)}'
      f' - a coin flip')
print(f'  mean AUC change {(best.auc - best.auc_full).mean():+.5f}')
print(f'  the best setting uses the full context in '
      f'{int((best.context_size == "full").sum())}/{len(best)} cases')
print(f'  median ratio of the best setting: {float((best.ctx / best.ep_context).median()):.2f}')
print('\nThis is the law, not an exception to it: reaching a matched ratio at full')
print('context needs episodes as long as the training set, and the longest episode')
print(f'in this grid is {int(T.ep_context.max())} context rows - where a T4 stops on')
print('tables with up to 110 features. At full context you are always off-diagonal.')

Best fine-tuned setting vs the base model at FULL context:
  -0.29% relative log-loss, winning 38/70 - a coin flip
  mean AUC change -0.00301
  the best setting uses the full context in 56/70 cases
  median ratio of the best setting: 3.27

This is the law, not an exception to it: reaching a matched ratio at full
context needs episodes as long as the training set, and the longest episode
in this grid is 1638 context rows - where a T4 stops on
tables with up to 110 features. At full context you are always off-diagonal.


## 6. Is the damage calibration, or lost knowledge?Temperature scaling is monotone per example in the binary case, so it leaves AUCexactly unchanged and moves only log-loss. If out-of-band damage were puremiscalibration, log-loss would degrade while AUC barely moved.

In [12]:

rows = []
for label, sub in [('matched (0.5 - 2)', T[(T.ratio >= .5) & (T.ratio <= 2)]),
                   ('extrapolating (> 4)', T[T.ratio > 4])]:
    ll = sub.groupby(['run', 'target']).rel.mean().mean()
    au = sub.groupby(['run', 'target']).d_auc.mean().mean()
    rows.append(dict(regime=label, rel_logloss=f'{ll:+.2f}%',
                     d_auc=f'{au:+.5f}', ratio_of_magnitudes=f'{abs(ll)/abs(au*100):.1f}x'))
print(pd.DataFrame(rows).to_string(index=False))
print('\nOut of band, log-loss degrades 7x more than AUC does, against ~2x inside it.')
print('Most of the damage is miscalibration rather than lost knowledge - which')
print('predicts that a single fitted temperature would recover much of it.')
print('That experiment is designed and not run; see README section 9.')

             regime rel_logloss    d_auc ratio_of_magnitudes
  matched (0.5 - 2)      +3.60% +0.01631                2.2x
extrapolating (> 4)      -4.06% -0.00558                7.3x

Out of band, log-loss degrades 7x more than AUC does, against ~2x inside it.
Most of the damage is miscalibration rather than lost knowledge - which
predicts that a single fitted temperature would recover much of it.
That experiment is designed and not run; see README section 9.


## 7. FigureThe figure the README opens with. One panel per deployment context, the same 37dataset-runs inside every panel.

In [13]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

S1, INK, INK2, MUTED, SURF = '#2a78d6', '#0b0b0b', '#52514e', '#b9b8b2', '#fcfcfb'
HARM = '#e8d5cc'

def panel(cs):
    s = C[C.context_size == cs]
    piv = s.pivot_table(index=['run', 'target'], columns='ep_context', values='d_auc')
    piv = piv[[e for e in EPS if e in piv.columns]].dropna()
    ratios = [float(s[s.ep_context == e].ctx.mean()) / e for e in piv.columns]
    return piv, ratios

data = {cs: panel(cs) for cs in CTX}
lo = min(p.min().min() for p, _ in data.values()) - .004
hi = max(p.max().max() for p, _ in data.values()) + .008

fig, axes = plt.subplots(2, 4, figsize=(15, 7.2), facecolor=SURF, sharey=True)
for ax, cs in zip(axes.ravel(), CTX):
    piv, ra = data[cs]
    ax.set_facecolor(SURF)
    ax.axhspan(lo, 0, color=HARM, alpha=.55, lw=0, zorder=0)
    x = list(piv.columns); y = [piv[e].mean() for e in x]
    ax.plot(x, y, color=S1, marker='o', ms=8, lw=2.2, zorder=4,
            markeredgecolor=SURF, markeredgewidth=1.5)
    for xi, yi, ri in zip(x, y, ra):
        ax.annotate(f'r={ri:.2f}', (xi, yi), xytext=(0, 11), textcoords='offset points',
                    color=INK2, fontsize=8, ha='center')
    ax.axhline(0, color=INK2, lw=1.2, ls=(0, (5, 3)), zorder=3)
    ax.set_xscale('log'); ax.set_xticks(EPS)
    ax.set_xticklabels([str(int(e)) for e in EPS], fontsize=9)
    ax.set_xlim(70, 2400); ax.set_ylim(lo, hi)
    ax.tick_params(colors=INK2, labelsize=9, length=3)
    ax.grid(axis='y', color=MUTED, alpha=.4, lw=.6); ax.set_axisbelow(True)
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_color(MUTED)
    lbl = 'full training set' if cs == 'full' else f'{cs} rows'
    ax.set_title(f'context {lbl}   (n={len(piv)})', fontsize=10.5, color=INK,
                 loc='left', pad=16)
for ax in axes[1]:
    ax.set_xlabel('fine-tuning episode context rows', fontsize=9.5, color=INK2)
for ax in axes[:, 0]:
    ax.set_ylabel('AUC gained by fine-tuning', fontsize=9.5, color=INK2)
fig.suptitle('One panel per deployment context. Same datasets within a panel, always.\n'
             'r = ratio at that point. Shaded = worse than not fine-tuning.',
             fontsize=12.5, color=INK, ha='left', x=.006, y=1.035, fontweight='medium')
fig.tight_layout(rect=[0, 0, 1, .95])
out = Path('figures') if Path('figures').is_dir() else Path('../figures')
fig.savefig(out / 'auc_by_context.png', dpi=160, bbox_inches='tight', facecolor=SURF)
print('written to', out / 'auc_by_context.png')

written to ../figures/auc_by_context.png


### The same data with the ratio on the x-axisThe figure the README opens with. The three curves share a shape but not a height:the ratio sets where the peak falls, the absolute size of the context sets how muchthere is to gain.

In [14]:

from matplotlib.ticker import NullFormatter
PAL = ['#2a78d6', '#eb6834', '#1baf7a']

def ratio_curve(ep):
    xs, ys = [], []
    for cs in CTX:
        g = C[(C.ep_context == ep) & (C.context_size == cs)]
        if len(g) < 6:
            continue
        xs.append(float(g.ctx.mean()) / ep)
        ys.append(float(g.groupby(['run', 'target']).d_auc.mean().mean()))
    return np.array(xs), np.array(ys)

fig, ax = plt.subplots(figsize=(10.2, 5.6), facecolor=SURF); ax.set_facecolor(SURF)
ax.axhspan(-.02, 0, color=HARM, alpha=.55, lw=0, zorder=0)
ends = []
for ep, c, mk in zip(EPS, PAL, ('o', 's', 'D')):
    x, y = ratio_curve(ep)
    lab = f'episode {int(ep)} rows'
    ax.plot(x, y, color=c, marker=mk, ms=7, lw=2.2, zorder=4, markeredgecolor=SURF,
            markeredgewidth=1.5, label=lab)
    k = int(np.argmax(y))
    ax.scatter([x[k]], [y[k]], s=170, facecolor='none', edgecolor=c, lw=1.7, zorder=5)
    ends.append([float(y[-1]), float(x[-1]), c, lab])
ax.axhline(0, color=INK2, lw=1.3, ls=(0, (5, 3)), zorder=3)
ax.set_xscale('log'); ax.set_xlim(.015, 90); ax.set_ylim(-.014, .032)
ylo, yhi = ax.get_ylim(); gap = (yhi - ylo) * .085
ends.sort(key=lambda r: r[0])
for i in range(1, len(ends)):
    if ends[i][0] - ends[i - 1][0] < gap:
        ends[i][0] = ends[i - 1][0] + gap
for y_, x_, c, lab in ends:
    ax.annotate(lab, (x_, y_), xytext=(9, 0), textcoords='offset points', color=c,
                fontsize=8.5, va='center', fontweight='medium',
                annotation_clip=False, zorder=6)
ax.set_xticks([.02, .05, .1, .25, .5, 1, 2, 4, 10, 20, 60])
ax.set_xticklabels(['0.02', '0.05', '0.1', '0.25', '0.5', '1', '2', '4', '10', '20', '60'],
                   fontsize=9.5)
ax.xaxis.set_minor_formatter(NullFormatter()); ax.tick_params(which='minor', length=0)
ax.tick_params(colors=INK2, labelsize=9.5, length=3)
ax.grid(axis='y', color=MUTED, alpha=.4, lw=.6); ax.set_axisbelow(True)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
for sp in ('left', 'bottom'): ax.spines[sp].set_color(MUTED)
ax.axvline(1, color=MUTED, lw=.9, zorder=1)
ax.set_xlabel('ratio    inference context / fine-tuning episode context',
              fontsize=10.5, color=INK2, labelpad=9)
ax.set_ylabel('AUC gained by fine-tuning', fontsize=10.5, color=INK2)
ax.annotate('no fine-tuning', (80, 0), xytext=(0, 6), textcoords='offset points',
            color=INK2, fontsize=9, ha='right', style='italic')
ax.text(.9, .0305, 'wasteful', fontsize=9.5, color=INK2, ha='right', va='top', style='italic')
ax.text(1.12, .0305, 'damaging', fontsize=9.5, color=INK2, ha='left', va='top', style='italic')
ax.annotate('below: worse than not fine-tuning at all', (.017, -.0115), color='#8c4a2f',
            fontsize=9.5, ha='left', va='center', fontweight='medium')
ax.legend(frameon=False, fontsize=9.5, loc='upper left', labelcolor=INK2, handlelength=1.8)
fig.suptitle('Every curve rises, peaks, then turns negative - but not at a common ratio',
             fontsize=13, color=INK, ha='left', x=.008, y=1.045, fontweight='medium')
fig.text(.008, .965, 'AUC peaks sit at ratios of 0.63, 0.25 and 0.08 - a factor of eight apart. '
         '24 OpenML binary tasks,\n200 steps, three runs, the same 37 dataset-runs in every '
         'point. Circles mark each peak.',
         fontsize=9.5, color=INK2, ha='left')
fig.tight_layout(rect=[0, 0, 1, .945])
fig.savefig(out / 'auc_vs_ratio.png', dpi=170, bbox_inches='tight', facecolor=SURF)
print('written to', out / 'auc_vs_ratio.png')

written to ../figures/auc_vs_ratio.png


### Where the three episodes actually fallThe figure that makes "bigger is not always better" visible: the winner is the episodenearest the optimum band, and in the lower rows all three sit to its right only because1638 context rows is the largest episode a T4 will hold.

In [15]:

from matplotlib.lines import Line2D
POS, NEG, BAND = '#2a78d6', '#e34948', '#dce9f8'

pts = []
for yi, cs in enumerate(CTX):
    for ep in EPS:
        g = C[(C.context_size == cs) & (C.ep_context == ep)]
        if len(g) < 6:
            continue
        pts.append(dict(y=len(CTX) - 1 - yi, cs=cs, ep=int(ep),
                        r=float(g.ctx.mean()) / ep,
                        v=float(g.groupby(['run', 'target']).d_auc.mean().mean())))
P = pd.DataFrame(pts)
best = {cs: P[P.cs == cs].loc[P[P.cs == cs].v.idxmax()] for cs in CTX if (P.cs == cs).any()}

fig, ax = plt.subplots(figsize=(11.4, 6.2), facecolor=SURF); ax.set_facecolor(SURF)
ax.axvspan(.5, 1.3, color=BAND, alpha=.75, lw=0, zorder=0)
ax.text(.8, len(CTX) - .35, 'where 6 of the 8 winners fall', fontsize=9.5, color='#1c5cab',
        ha='center', style='italic', fontweight='medium')
for yi in range(len(CTX)):
    ax.axhline(yi, color=MUTED, alpha=.35, lw=.7, zorder=1)
smax = P.v.abs().max()
for _, r in P.iterrows():
    size = 60 + 900 * abs(r.v) / smax
    ax.scatter([r.r], [r.y], s=size, color=(POS if r.v > 0 else NEG), alpha=.9,
               zorder=4, edgecolor=SURF, linewidth=1.4)
    if r.ep == best[r.cs].ep:
        ax.scatter([r.r], [r.y], s=size + 260, facecolor='none', edgecolor=INK,
                   lw=1.5, zorder=5)
    rad = np.sqrt(size) / 2.0
    ax.annotate(f'{r.ep}', (r.r, r.y), xytext=(0, -(rad + 9.5)),
                textcoords='offset points', color=INK2, fontsize=7.8,
                ha='center', va='top', zorder=6)
ax.set_yticks(range(len(CTX)))
ax.set_yticklabels([('full train' if c == 'full' else c) for c in CTX][::-1], fontsize=10)
ax.set_xscale('log'); ax.set_xlim(.014, 90); ax.set_ylim(-.95, len(CTX) - .15)
ax.set_xticks([.02, .05, .1, .25, .5, 1, 2, 4, 10, 20, 60])
ax.set_xticklabels(['0.02', '0.05', '0.1', '0.25', '0.5', '1', '2', '4', '10', '20', '60'],
                   fontsize=9.5)
ax.xaxis.set_minor_formatter(NullFormatter()); ax.tick_params(which='minor', length=0)
ax.tick_params(colors=INK2, labelsize=9.5, length=3)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
for sp in ('left', 'bottom'): ax.spines[sp].set_color(MUTED)
ax.set_xlabel('ratio    inference context / fine-tuning episode context   '
              '(number under each dot = episode length in rows)',
              fontsize=10, color=INK2, labelpad=10)
ax.set_ylabel('deployment context, rows', fontsize=10.5, color=INK2)
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=POS, ms=9, label='fine-tuning helps'),
                   Line2D([], [], marker='o', ls='', color=NEG, ms=9, label='fine-tuning hurts'),
                   Line2D([], [], marker='o', ls='', mfc='none', mec=INK, ms=12,
                          label='best of the three')],
          frameon=False, fontsize=9.5, loc='upper right', labelcolor=INK2,
          handletextpad=.6, bbox_to_anchor=(1.0, 1.02))
fig.suptitle('No single episode length wins everywhere - the winner moves with the context',
             fontsize=13, color=INK, ha='left', x=.008, y=1.05, fontweight='medium')
fig.text(.008, .965, 'Dot size = size of the effect. Ringed = best of the three at that context. '
         'Below 256 rows the shortest episode wins,\nfrom 1024 rows up the longest does - '
         'and 1638 rows is simply the longest we could fit on a T4.',
         fontsize=9.5, color=INK2, ha='left', linespacing=1.5)
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(out / 'where_the_episodes_fall.png', dpi=170, bbox_inches='tight', facecolor=SURF)
print('written to', out / 'where_the_episodes_fall.png')

written to ../figures/where_the_episodes_fall.png


### What the ratio does and does not captureThe ratio summarises relative log-loss reasonably and AUC poorly. The table below isthe honest version of the claim: a longer episode does push the sign change out tolarger contexts, but sub-proportionally, so the crossing *ratio* is not a constant.

In [16]:

print(f"{'episode':>8} {'AUC peak':>22} {'log-loss peak':>24} {'AUC crosses zero':>26}")
for ep in EPS:
    xs, ya, yl = [], [], []
    for cs in CTX:
        g = C[(C.ep_context == ep) & (C.context_size == cs)]
        if len(g) < 6:
            continue
        xs.append((float(g.ctx.mean()) / ep, float(g.ctx.mean())))
        ya.append(float(g.groupby(['run', 'target']).d_auc.mean().mean()))
        yl.append(float(g.groupby(['run', 'target']).rel.mean().mean()))
    ka, kl = int(np.argmax(ya)), int(np.argmax(yl))
    cross = None
    for i in range(len(ya) - 1):
        if ya[i] > 0 >= ya[i + 1]:
            t = ya[i] / (ya[i] - ya[i + 1])
            cross = np.exp(np.log(xs[i][1]) + t * (np.log(xs[i + 1][1]) - np.log(xs[i][1])))
            break
    c_txt = '--' if cross is None else f'ctx={cross:>6.0f}  r={cross/ep:>5.2f}'
    print(f'{int(ep):>8}   r={xs[ka][0]:>5.2f} ctx={xs[ka][1]:>5.0f}'
          f'      r={xs[kl][0]:>5.2f} ctx={xs[kl][1]:>5.0f}      {c_txt}')
print('\nThe crossing moves out with episode length - 856, 1454, 2251 rows - which is')
print('the robust claim. It moves sub-proportionally: a 16x longer episode buys 2.6x')
print('more deployable context, so the crossing ratio falls from 8.4 to 1.4 rather')
print('than staying put. Use the ratio to locate the regime, not as a constant.')

 episode               AUC peak            log-loss peak           AUC crosses zero
     102   r= 0.63 ctx=   64      r= 0.63 ctx=   64      ctx=   856  r= 8.39
     512   r= 0.25 ctx=  128      r= 0.50 ctx=  256      ctx=  1454  r= 2.84
    1638   r= 0.08 ctx=  128      r= 0.31 ctx=  512      ctx=  2251  r= 1.37

The crossing moves out with episode length - 856, 1454, 2251 rows - which is
the robust claim. It moves sub-proportionally: a 16x longer episode buys 2.6x
more deployable context, so the crossing ratio falls from 8.4 to 1.4 rather
than staying put. Use the ratio to locate the regime, not as a constant.


## What this notebook establishes, and what it does not**Established.** Fine-tuning on episodes much shorter than the deployment contextdegrades the model relative to not fine-tuning at all. The context at which thegain changes sign moves out as episodes lengthen - 856, 1454 and 2251 rows forthe three lengths here. No single episode length is best at every context, andthe longest is never best at small ones. The effect survives an equal-supervisioncontrol and a second seed.**Not established.** Any mechanism. That the ratio is the governing variable: itsummarises relative log-loss well and AUC badly, and the crossing ratio falls from8.4 to 1.4 as episodes lengthen rather than holding constant. Any single threshold.Ratios below 0.1 are only reachable with contexts of 128 rows or fewer, so the farleft of the curve is confounded with small absolute contexts. Binary classificationonly, one model family, one benchmark suite, 200 gradient steps.